In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from qdrant_client import QdrantClient, models

from audio.conversion import conversion
from audio.processing import resample, denoise, vad
from ai.embedding import embedding
import torch

from concurrent.futures import ProcessPoolExecutor

import os

os.environ["OMP_NUM_THREADS"] = "12"
os.environ["MKL_NUM_THREADS"] = "12"

torch.set_num_threads(12)

c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
VOXFORGE_PATH = r"C:\Users\PC5\Documents\Voice-Recognition\voxforge-fr"
COLLECTION_NAME = "voice_data_base"

client = QdrantClient(host="localhost", port=6333)

In [3]:
def clean_embedding(audio_bytes: bytes):

    raw = conversion(audio_bytes)

    audio, sr = resample(raw)

    audio = denoise(audio, sr)

    audio, issue = vad(audio, sr)

    if issue[0]:
        return None

    return embedding(audio)

In [4]:
def parse_readme(readme_path):

    metadata = {
        "username": None,
        "gender": None,
        "age": None,
        "dialect": None,
    }

    if not readme_path.exists():
        return metadata

    with open(readme_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    for line in content.splitlines():

        if line.startswith("User Name:"):
            metadata["username"] = (
                line.replace("User Name:", "").strip()
            )

        elif line.startswith("Gender:"):
            metadata["gender"] = (
                line.replace("Gender:", "").strip()
            )

        elif line.startswith("Age Range:"):
            metadata["age"] = (
                line.replace("Age Range:", "").strip()
            )

        elif line.startswith("Pronunciation dialect:"):
            metadata["dialect"] = (
                line.replace(
                    "Pronunciation dialect:",
                    ""
                ).strip()
            )

    return metadata

In [5]:
speakers = {}

for folder in Path(VOXFORGE_PATH).iterdir():

    if not folder.is_dir():
        continue

    wav_dir = folder / "wav"
    readme = folder / "etc" / "README"

    if not wav_dir.exists():
        continue

    metadata = parse_readme(readme)

    username = metadata["username"]

    if username is None:
        continue

    wavs = sorted(list(wav_dir.glob("*.wav")))

    if len(wavs) < 2:
        continue

    if username not in speakers:

        speakers[username] = {
            "wavs": [],
            "gender": metadata["gender"],
            "age": metadata["age"],
            "dialect": metadata["dialect"],
        }

    speakers[username]["wavs"].extend(wavs)

print("Speakers:", len(speakers))

Speakers: 332


TESTS

In [6]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

all_tasks = []

for username, data in speakers.items():

    wavs = sorted(list(set(data["wavs"])))

    if len(wavs) < 2:
        continue

    test_wavs = wavs[1:]

    for wav_file in test_wavs:

        all_tasks.append(
            (
                username,
                wav_file,
                data["gender"],
                data["age"],
                data["dialect"],
            )
        )

print("TOTAL TESTS:", len(all_tasks))

def process_test(task):

    username, wav_file, gender, age, dialect = task

    try:

        with open(wav_file, "rb") as f:
            audio_bytes = f.read()

        emb = clean_embedding(audio_bytes)

        if emb is None:
            return None

        result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=emb,
            limit=1,
            search_params=models.SearchParams(
                hnsw_ef=200,
                exact=False,
            ),
        )

        if len(result.points) == 0:
            return None

        point = result.points[0]

        predicted = point.payload["username"]

        score = float(point.score)

        correct = predicted == username

        return {
            "speaker": username,
            "predicted": predicted,
            "score": score,
            "correct": correct,
            "gender": gender,
            "age": age,
            "dialect": dialect,
        }

    except Exception as e:
        print("ERROR:", wav_file, e)
        return None

TOTAL TESTS: 23420


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

results = []

tasks = []

for username, data in speakers.items():

    wavs = sorted(list(set(data["wavs"])))

    if len(wavs) < 2:
        continue

    test_wavs = wavs[1:]

    for wav_file in test_wavs:
        tasks.append((username, data, wav_file))


def process_test(task):

    username, data, wav_file = task

    with open(wav_file, "rb") as f:
        audio_bytes = f.read()

    emb = clean_embedding(audio_bytes)

    if emb is None:
        return None

    result = client.query_points(
        collection_name=COLLECTION_NAME,
        query=emb,
        limit=1,
    )

    if len(result.points) == 0:
        return None

    point = result.points[0]

    predicted = point.payload["username"]

    return {
        "speaker": username,
        "predicted": predicted,
        "score": float(point.score),
        "correct": predicted == username,
        "gender": data["gender"],
        "age": data["age"],
        "dialect": data["dialect"],
    }


with ThreadPoolExecutor(max_workers=1) as executor:

    futures = [executor.submit(process_test, task) for task in tasks]

    for future in tqdm(
        as_completed(futures), total=len(futures), desc="Testing speakers"
    ):
        r = future.result()

        if r is not None:
            results.append(r)

results_df = pd.DataFrame(results)

results_df.to_csv("voxforge_results.csv", index=False)

print("DONE")

Testing speakers: 100%|██████████| 23420/23420 [6:00:09<00:00,  1.08it/s]   

DONE


Accuracy

In [8]:
results_df["correct"].mean()

0.6508574605482615

Accuracy per gender

In [9]:
results_df.groupby("gender")["correct"].mean()

Unexpected exception formatting exception. Falling back to standard exception


C:\Users\PC5\AppData\Local\Programs\Python\Python310\lib\inspect.py:869: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
Traceback (most recent call last):
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\PC5\AppData\Local\Temp\ipykernel_13480\1961562351.py", line 1, in <module>
    results_df.groupby("gender")["correct"].mean()
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\pandas\core\frame.py", line 9210, in groupby
    return DataFrameGroupBy(
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\pandas\core\groupby\groupby.py", line 1331, in __init_

Accuracy per dialect

In [10]:
results_df.groupby("dialect")["correct"].mean()

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\PC5\AppData\Local\Temp\ipykernel_13480\1207108228.py", line 1, in <module>
    results_df.groupby("dialect")["correct"].mean()
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\pandas\core\frame.py", line 9210, in groupby
    return DataFrameGroupBy(
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\pandas\core\groupby\groupby.py", line 1331, in __init__
    grouper, exclusions, obj = get_grouper(
  File "c:\Users\PC5\Documents\Voice-Recognition\.venv\lib\site-packages\pandas\core\groupby\grouper.py", line 1043, in get_grouper
    raise KeyError(gpr)
KeyError: 'dialect'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\PC5\Documents\Voic

Score distribution

In [ ]:
results_df["score"].hist(bins=50)

plt.xlabel("Similarity score")
plt.ylabel("Count")
plt.title("Score distribution")

plt.show()

Best threshold

In [ ]:
best_threshold = 0
best_acc = 0

for threshold in [x / 100 for x in range(0, 100)]:

    accepted = results_df["score"] > threshold

    acc = (accepted == results_df["correct"]).mean()

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print(best_threshold)
print(best_acc)